Regression

In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import getpass
import gc
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
a = str(device).title()
print(f"Using Device = {a}")


script_dir = os.path.dirname(os.path.abspath(os.getcwd()))

df_path = os.path.join(script_dir,"Files","Student Performance Score Dataset.csv")
df = pd.read_csv(df_path)

df = df.dropna()

x = df.drop(columns="final_score").values
y = df["final_score"].values.reshape(-1,1)

x_scaler = StandardScaler()
y_scaler = StandardScaler()
x = x_scaler.fit_transform(x)
y = y_scaler.fit_transform(y)

x_train, x_test, y_train ,y_test = train_test_split(x,y, test_size=0.2, random_state=23)
num_features = x_train.shape[1]

x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

print(f"X = {x.shape}")
print(f"Y = {y.shape}")
print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")

user_account = getpass.getuser()
predictions_bar_chart = os.path.join(script_dir,"Graphs",f"Student Performance Score Regressor Network Predictions Bar Chart.png")

class Regressor(nn.Module):
    def __init__(self, input_dim =num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=8),
            nn.LeakyReLU(),
            nn.BatchNorm1d(8),

            nn.Linear(in_features=8, out_features=16),
            nn.LeakyReLU(),
            nn.BatchNorm1d(16),
            
            nn.Linear(in_features=16, out_features=28),
            nn.LeakyReLU(),
            nn.BatchNorm1d(28),

            nn.Linear(in_features=28, out_features=47),
            nn.LeakyReLU(),
            nn.BatchNorm1d(47),
            nn.Dropout(p=0.7),   
            
            nn.Linear(in_features=47, out_features=52),
            nn.ELU(),
            nn.BatchNorm1d(52),
            nn.Dropout(p=0.8),   

            nn.Linear(in_features=52, out_features=68),
            nn.ELU(),
            nn.BatchNorm1d(68),
            nn.Dropout(p=0.3),

            nn.Linear(in_features=68, out_features=42),
            nn.LeakyReLU(),
            nn.BatchNorm1d(42),   

            nn.Linear(in_features=42, out_features=21),
            nn.LeakyReLU(),
            nn.BatchNorm1d(21),

            nn.Linear(in_features=21, out_features=1)
        )

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)

def run_grid_search(rates_list, epochs_list, seed_val):
    best_loss = float("inf")
    best_state = None
    loss_fn = nn.L1Loss()

    print(f"\n== Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<8} | {'Val Loss':<10}")
    print(f"="*65)
    losses = []

    start_time = time.perf_counter()

    for lr in rates_list:
        for epoch in epochs_list:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))

            model = Regressor().to(device)   
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            for e in tqdm(range(epoch)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds, y_train)     

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_loss = loss_fn(model(x_test), y_test)
                losses.append(loss)

            print(f"{lr:<15} | {epoch:<8} | {loss:<10.4f}")

            if test_loss < best_loss:
                best_loss = test_loss
                import copy
                best_state = copy.deepcopy(model.state_dict()) 

            del model, optimizer
            gc.collect()       
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    end_time = time.perf_counter()
    execution_time = end_time - start_time
    print(f"\n● Best Loss Achieved = {best_loss}")  
    print(f"● Worst Loss = {max(losses)}")        
    print(f"● Training Time = {execution_time} Seconds\n")          
  

    return best_state, best_loss,losses

epochs_list = [380, 700, 910, 840, 1000, 220, 590]
rates = [0.1, 0.01, 0.001, 0.2, 0.10, 0.00001, 0.3]

for file in [predictions_bar_chart, f"C:/Users/{user_account}/OneDrive/Desktop/Student Performance Score Regressor Network Details.txt", f"C:/Users/{user_account}/OneDrive/Desktop/Student Performance Score Regressor Network Parameters.pth"]:
    if os.path.exists(file):
        try:os.remove(file)
        except OSError:pass

seed = float(input("-- Enter a Value for a Seed : ").strip())
best_state, best_loss, losses = run_grid_search(rates, epochs_list, seed)
fresh_model = Regressor().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

ac = {
    "Yes":1,
    "No":0
}

session_inputs = []
session_predictions = []

while True:
    raw_name_input = input("-- Enter your Name ( or type 'exit' to stop ): ").strip()
    
    if raw_name_input.lower() == 'exit' or not raw_name_input:
        print("\nExiting Interactive Loop ...")
        break
        
    user_name = raw_name_input.title()
    
    hours_studied = float(input("-- How many Hours did you Study for : ").strip())
    attendance_rate = float(input("-- What is your Attendance Rate : ").strip())
    previous_score = float(input("-- What was your Previous Score : ").strip())
    sleep_hours = float(input("-- How many Hours did you Sleep for : ").strip())
    extracurricular_activities = input("-- Do you do any Extra Curricular Activities (Yes/No) : ").strip().title()
    internet_quality = float(input("-- Enter your Internet Quality : ").strip())

    ex = ac.get(extracurricular_activities, 0)

    raw_array = np.array([[hours_studied, attendance_rate, previous_score, sleep_hours, ex, internet_quality]])
    scaled_array = x_scaler.transform(raw_array)
    scaled_tensor = torch.tensor(scaled_array, dtype=torch.float32).to(device)

    with torch.inference_mode():
        preds = fresh_model(scaled_tensor).cpu().numpy()

    untransformed = y_scaler.inverse_transform(preds).item()
    print(f"Predicted Performance Score for you, {user_name} = {untransformed:.2f}")   

    session_inputs.append(f"{user_name} ({round(hours_studied)} Hours Studied )")
    session_predictions.append(untransformed)

onedrive = input("-- Do you use One Drive or Not ( Yes / No ) :").strip().title()
if onedrive == "Yes":
    desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Student Performance Score Regressor Network Details.txt"
elif onedrive == "No":
    desktop_path = f"C:/Users/{user_account}/Desktop/Student Performance Score Regressor Network Details.txt"

try:
    with open(desktop_path, "w", encoding="utf-8") as file:
            file.write(f"== Student Performance Score Regressor Network Weights & Biases ==\n")   
            file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.3f}\n")     
            file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.3f}\n")     
            file.write(f"● Best Loss = {best_loss:.3f}\n")
            file.write(f"● Worst Loss = {max(losses):.3f}\n")
    print(f"\nSuccessefully Saved the Network Details")
except BaseException as e:
    print(f"Failed to Save Network Details = {e}")       

Image Classification

In [ ]:
import matplotlib
matplotlib.use("Agg")
import os, copy, gc, torch
from PIL import Image
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torchmetrics.classification import MulticlassAccuracy
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
train_data = os.path.join(script_dir, "Files","Image Classification","Train")
test_data = os.path.join(script_dir, "Files","Image Classification","Test")

train_pipeline = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(p=0.6),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.589, 0.617], std=[0.229, 0.213, 0.286])
])
test_pipeline = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[1.9, 0.45, 0.68], std=[0.348, 0.112, 0.982])
])
train_dataset = datasets.ImageFolder(root=train_data, transform=train_pipeline)
test_dataset = datasets.ImageFolder(root=test_data, transform=test_pipeline)

batch_size = 50
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False, pin_memory=True)

classnames = train_dataset.classes
num_classes = len(classnames)

class ComputerVision(nn.Module):
    def __init__(self, in_channels:int = 3, num_classes:int = num_classes):
        super().__init__()
        self.features = nn.Sequential(

            nn.Conv2d(in_channels=in_channels, out_channels=32, kernel_size=4, stride=3, padding=2),
            nn.BatchNorm1d(35),
            nn.LeakyReLU(negative_slope=0.93),
            nn.MaxPool2d(kernel_size=2.3, stride=1.6),

            nn.Conv2d(in_channels=35, out_channels=87, kernel_size=5, stride=3, padding=6),
            nn.BatchNorm1d(87),
            nn.LeakyReLU(negative_slope=0.82),
            nn.MaxPool2d(kernel_size=2, stride=3.1),

            nn.Conv2d(in_channels=89, out_channels=126, kernel_size=9, stride=4, padding=10),
            nn.LeakyReLU(negative_slope=1.3),
            nn.BatchNorm1d(126),
            nn.MaxPool2d(kernel_size=3.6, stride=5.2),

            nn.Conv2d(in_channels=126, out_channels=256, kernel_size=2, stride=3, padding=1.5),
            nn.LeakyReLU(negative_slope=0.18),
            nn.BatchNorm1d(256),

            nn.AdaptiveAvgPool1d((1,1))
        )

        self.classfier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=256, out_features=64),
            nn.LeakyReLU(negative_slope=0.45),
            nn.BatchNorm1d(64),
            nn.Dropout(p=0.4),

            nn.Linear(in_features=64, out_features=num_classes)
        )

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classfier(x)
        return x

losses = []
accuracies = []

def grid_search(epoch_count_list, learning_rates):
    best_state = None

    for lr in learning_rates:
        for count in epoch_count_list:
            print(f"Evaluating Configuration : Lr = {lr} | Epochs = {count}")
            model = ComputerVision().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            loss_function = nn.CrossEntropyLoss().to(device)
            accuracy_metric = MulticlassAccuracy(num_classes=num_classes).to(device)

            for e in tqdm(range(count)):
                model.train()
                for image, labels in train_loader:
                    outputs = model(image)
                    loss = loss_function(outputs, labels)

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                model.eval()
                with torch.inference_mode():
                    for image, labels in test_loader:
                        test_outputs = model(image.to(device))
                        preds = torch.argmax(test_outputs, dim=1)
                        test_acc = accuracy_metric(preds.to(device), labels.to(device))
                        test_loss = loss_function(preds.to(device), labels.to(device))
                        losses.append(test_loss)
                        accuracies.append(test_acc)

            best_state = copy.deepcopy(model.state_dict())      
            del model, optimizer, loss_function, accuracy_metric
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss = {min(losses):.2f} | Best Accuracy = {max(accuracies):.2f} %")
    print(f"● Worst Loss = {max(losses):.2f} | Worst Accuracy = {min(accuracies):.2f} %")   
    return best_state    

epoch_counts_list = [980, 1230, 780, 810]
rates_list = [0.01, 0.1, 0.001, 0.2]
best_state = grid_search(epoch_count_list=epoch_counts_list, learning_rates=rates_list)

fresh_model = ComputerVision().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

inference_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.987, 0.456, 0.213], std=[0.115, 0.581, 0.692])
])

while True:
    user_image_path = input("-- Enter the Absolute Path to an Image File ( or type 'exit' to quit ) : ").strip()
    if user_image_path.lower() == "exit":
        print(f"Exiting Interactive Loop ...")
        break

    if not os.path.exists(user_image_path):
        print(f"File Path does not Exist")
        continue

    try:
        raw_image = Image.open(user_image_path).convert("RGB")
        inputs_tensor = inference_transform(raw_image).unsqueeze(0).to(device)

        with torch.inference_mode():
            logits = fresh_model(inputs_tensor)
            preds = torch.softmax(logits, dim=1)
            pred_idx = torch.argmax(preds, dim=1).item()
            confidence = preds[pred_idx].item() * 100

        print(f"\n● Predicted Class = {classnames[pred_idx].title()}")
        print(f"● Confidence Score = {confidence:.2f}")

    except BaseException as e:
        print(f"Failed to Process Image File | Error has Occured = {e}")     

try:
    plt.figure(figsize=(15,4))
    plt.imshow(inputs_tensor)
    plt.title(f"Prediction = {classnames[pred_idx].title()} | Confidence = {confidence:.2f}")
except BaseException as e:
    print(f"Failed to Graph your Image")